In [8]:
import pandas as pd
import os

def load_excel_dataset(file_path):
    # Читаем эксель
    df = pd.read_excel(file_path)
    
    # Подождите, дайте проверю колонки... 
    # Если вы копировали мой прошлый список, то колонки должны быть такие:
    required_columns = ['wav_path', 'text', 'emotion']
    
    if not all(col in df.columns for col in required_columns):
        # О, я упустил, что названия могут отличаться. 
        # Если в Excel колонки называются иначе, поправьте их здесь:
        print(f"Внимание! В файле найдены колонки: {list(df.columns)}")
        print(f"Ожидались: {required_columns}")
        return None

    # Очищаем текст от лишних пробелов и пустых строк
    df['text'] = df['text'].astype(str).str.strip()
    df = df[df['text'] != '']
    
    return df

# Теперь вызываем вот так
path_to_file = "/kaggle/input/datasets/seprenbolat/ozon-test/ozon_test.xlsx"
dataset_df = load_excel_dataset(path_to_file)

if dataset_df is not None:
    print(f"Загружено строк: {len(dataset_df)}")
    display(dataset_df.head())

Загружено строк: 15


,wav_path,text,emotion
0,wavs/ozon_001.wav,Здравствуйте! Поздравляю вас с регистрацией на...,neutral
1,wavs/ozon_002.wav,Я только что зарегистрировался и пока не подкл...,neutral
2,wavs/ozon_003.wav,"Подождите, какой счет вы имеете в виду? Я нови...",questioning
3,wavs/ozon_004.wav,Оставьте меня в покое! Вы уже третий раз сегод...,anger
4,wavs/ozon_005.wav,Хватит! Я не могу больше переносить вашу манип...,anger


In [9]:
import pandas as pd

# Загружаем наш исходный эксель
df = pd.read_excel("/kaggle/input/datasets/seprenbolat/ozon-test/ozon_test.xlsx")

# Готовим данные для метаданных
# Формат: путь_к_файлу|текст|id_эмоции
metadata = []
emotion_map = {
    'neutral': 0,
    'questioning': 1,
    'admiration': 2,
    'anger': 3,
    'sadness': 4,
    'surprise': 5,
    'fear': 6,
    'disgust': 7,
    'sarcasm': 8
}

for i, row in df.iterrows():
    emotion = str(row['emotion']).lower().strip()
    emo_id = emotion_map.get(emotion, 0)
    # Название файла должно совпадать с тем, что мы генерировали ранее
    file_path = f"sample_{i}_{emotion}.wav"
    text = str(row['text']).replace("|", " ") # Убираем разделители из текста, чтобы не сломать CSV
    
    metadata.append(f"{file_path}|{text}|{emo_id}")

# Сохраняем в текстовый файл
with open("metadata.csv", "w", encoding="utf-8") as f:
    for line in metadata:
        f.write(line + "\n")

print("Файл metadata.csv успешно создан!")


Файл metadata.csv успешно создан!


In [10]:
import json

emotions_config = {
    "emotions": emotion_map,
    "sample_rate": 48000,
    "language": "ru"
}

with open("emotions_config.json", "w", encoding="utf-8") as f:
    json.dump(emotions_config, f, ensure_ascii=False, indent=4)

print("Конфиг emotions_config.json готов!")


Конфиг emotions_config.json готов!


In [11]:
import pandas as pd

# Наш словарь ударений для максимальной "человечности"
accents_map = {
    "Здравствуйте!": "Здр+авствуйте!",
    "Поздравляю вас с регистрацией на Ozon.": "Поздравл+яю вас с регистр+ацией на Оз+он.",
    "Я только что зарегистрировался и пока не подключил свой расчетный счет.": "Я т+олько что зарегистр+ировался и пок+а не подкл+ючил свой расч+етный счет.",
    "Подождите, какой счет вы имеете в виду?": "Подожд+ите, как+ой счет вы им+еете в вид+у?",
    "Оставьте меня в покое!": "Ост+авьте мен+я в пок+ое!",
    "Хватит!": "Хв+атит!",
    "Могли бы вы мне рассказать подробнее о возможностях оптимизации операций?": "Могл+и бы вы мне рассказ+ать подр+обнее о возм+ожностях оптимиз+ации опер+аций?",
    "Очень полезно.": "+Очень пол+езно.",
    "Вау, этот интерфейс просто невероятен!": "В+ау, +этот интерф+ейс пр+осто неверо+ятен!",
    "Неужели можно так просто подключить счет?": "Неуж+ели м+ожно так пр+осто подкл+ючить счет?",
    "Благодарю вас за доверие!": "Благодар+ю вас за дов+ерие!"
}

# Читаем ваш эксель
df = pd.read_excel("/kaggle/input/datasets/seprenbolat/ozon-test/ozon_test.xlsx")

metadata = []
emotion_map = {'neutral': 0, 'questioning': 1, 'admiration': 2, 'anger': 3, 'sadness': 4, 'surprise': 5, 'fear': 6, 'disgust': 7, 'sarcasm': 8}

for i, row in df.iterrows():
    text = str(row['text']).strip()
    emotion = str(row['emotion']).lower().strip()
    
    # Пытаемся найти фразу в нашем словаре ударений, если нет - берем оригинал
    accented_text = accents_map.get(text, text)
    emo_id = emotion_map.get(emotion, 0)
    file_path = f"sample_{i}_{emotion}.wav"
    
    metadata.append(f"{file_path}|{accented_text}|{emo_id}")

with open("metadata.csv", "w", encoding="utf-8") as f:
    for line in metadata:
        f.write(line + "\n")

print("Тема с подготовкой данных закрыта. metadata.csv готов!")


Тема с подготовкой данных закрыта. metadata.csv готов!


In [12]:
pip install coqui-tts   

  Using cached coqui_tts-0.27.5-py3-none-any.whl.metadata (21 kB)
Using cached coqui_tts-0.27.5-py3-none-any.whl (862 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 89.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 96.5 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.2.0
    Uninstalling hf-xet-1.2.0:
      Successfully uninstalled hf-xet-1.2.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformer

### XTTS-v2

In [13]:
# 1. Удаляем всё, что может мешать
!pip uninstall -y TTS coqui-tts transformers pydantic

# 2. Ставим всё строго определенных версий
# Мы ставим coqui-tts через --no-deps, чтобы она не тянула за собой "вражеские" обновления
!pip install coqui-tts==0.22.0 --no-deps
!pip install transformers==4.38.0
!pip install pydantic==1.10.13
!pip install trainer
!pip install s3fs

# 3. Ставим остальные важные штуки, которые нужны для работы XTTS
!pip install mecab-python3 unidic-lite

Found existing installation: coqui-tts 0.27.5
Uninstalling coqui-tts-0.27.5:
  Successfully uninstalled coqui-tts-0.27.5
Found existing installation: transformers 5.3.0
Uninstalling transformers-5.3.0:
  Successfully uninstalled transformers-5.3.0
Found existing installation: pydantic 1.10.13
Uninstalling pydantic-1.10.13:
  Successfully uninstalled pydantic-1.10.13
ERROR: Ignored the following versions that require a different python version: 0.22.1 Requires-Python <3.12,>=3.9.0; 0.23.0 Requires-Python <3.12,>=3.9.0
ERROR: Could not find a version that satisfies the requirement coqui-tts==0.22.0 (from versions: 0.23.1, 0.24.0, 0.24.1, 0.24.2, 0.24.3, 0.25.0, 0.25.1, 0.25.2, 0.25.3, 0.26.0, 0.26.1, 0.26.2, 0.27.0, 0.27.1, 0.27.2, 0.27.3, 0.27.4, 0.27.5)
ERROR: No matching distribution found for coqui-tts==0.22.0
  Using cached transformers-4.38.0-py3-none-any.whl.metadata (131 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.15.2-c

In [14]:
import pandas as pd
import os
import torch
from TTS.api import TTS

# Теперь импорт должен пройти успешно
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

# Загружаем модель (при первом запуске скачает около 2 Гб)
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

# Путь к вашему экселю в Kaggle
path_to_excel = "/kaggle/input/datasets/seprenbolat/ozon-test/ozon_test.xlsx"
df = pd.read_excel(path_to_excel)

os.makedirs("output_audio", exist_ok=True)

for i, row in df.iterrows():
    text = str(row['text'])
    emotion = str(row['emotion']).lower().strip()
    
    # Путь к итоговому файлу
    out_file = f"output_audio/sample_{i}_{emotion}.wav"
    
    # Ищем референс для клонирования голоса и эмоции
    ref_path = f"references/ref_{emotion}.wav"
    
    if not os.path.exists(ref_path):
        # Если файла нет, попробуем взять хотя бы нейтральный голос
        ref_path = "references/ref_neutral.wav"
        if not os.path.exists(ref_path):
             print(f"Пропускаем строку {i}: нет аудио-референса для {emotion}")
             continue

    try:
        tts.tts_to_file(
            text=text,
            speaker_wav=ref_path,
            language="ru",
            file_path=out_file
        )
        print(f"Готово: {out_file}")
    except Exception as e:
        print(f"Ошибка на фразе '{text[:20]}...': {e}")

print("Обработка завершена. Все файлы в папке output_audio.")


ModuleNotFoundError: No module named 'TTS'

### Mozilla TTS

### ChatTTS

### MeloTTS

### Coqui TTS

### Bark TTS

In [29]:
# Ставим пакет, который объединяет всё нужное
!pip install f5-tts --quiet

# Нам также понадобится библиотека для работы с аудио
!pip install pydub librosa --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 48.2 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.7 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 47.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 53.9 MB/s eta 0:00:00
   ━

In [30]:
from f5_tts.api import F5TTS

# Инициализируем модель (она сама скачает веса, около 1 Гб)
f5tts = F5TTS()

# Пример генерации с эмоцией
# Мы берем один из твоих файлов Silero как "маяк" интонации
ref_audio = "/kaggle/working/dataset/wavs/sample_3_anger.wav" 
ref_text = "Хватит! Я не хочу никакой помощи от вас!" # Текст, который звучит в референсе

gen_text = "Слушайте, я требую немедленно закрыть мой расчетный счет!"

# Генерируем "человеческий" ответ
f5tts.infer(
    ref_file=ref_audio,
    ref_text=ref_text,
    gen_text=gen_text,
    file_wave="final_anger_result.wav"
)

print("Голос сгенерирован с учетом эмоции референса!")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
2026-03-11 23:29:37.882603: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773271778.059005      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN whe

Download Vocos from huggingface charactr/vocos-mel-24khz


config.yaml:   0%|          | 0.00/461 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/54.4M [00:00<?, ?B/s]

F5TTS_v1_Base/model_1250000.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]


vocab :  /usr/local/lib/python3.12/dist-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 

Converting audio...
Using custom reference text...

ref_text   Хватит! Я не хочу никакой помощи от вас!. 
gen_text 0 Слушайте, я требую немедленно закрыть мой расчетный счет!


Generating audio in 1 batches...


100%|██████████| 1/1 [00:04<00:00,  4.09s/it]

Голос сгенерирован с учетом эмоции референса!


In [31]:
import librosa
import soundfile as sf
import numpy as np

# Загружаем оригинал от Silero (тот, что был нейтральным)
y, sr = librosa.load("/kaggle/working/silero_audio/sample_6_neutral.wav")

# 1. Ускоряем СЛЕГКА (было 1.5, ставим 1.12)
# Это даст динамику, но сохранит дикцию
y_fast = librosa.effects.time_stretch(y, rate=1.12)

# 2. Добавим "агрессии" через усиление средних частот (эффект крика)
# Просто немного поднимем громкость, чтобы появились "пики"
y_angry = y_fast * 1.2

# 3. Сохраняем новый, более разборчивый референс
sf.write("clear_anger_ref.wav", y_angry, sr)

print("Новый референс готов. Он медленнее и четче.")

Новый референс готов. Он медленнее и четче.


In [32]:
import librosa
import soundfile as sf
import numpy as np

# Загружаем наш "слишком правильный" файл
y, sr = librosa.load("/kaggle/working/clear_anger_ref.wav")

# 1. Поднимаем скорость до оптимальной (1.25x)
y_human = librosa.effects.time_stretch(y, rate=1.15) # Ускоряем еще на 15% от текущего

# 2. Небольшой трюк для "жизни": меняем высоту тона на 0.5 полутона
# Это уберет эффект "замороженного" голоса
y_human = librosa.effects.pitch_shift(y_human, sr=sr, n_steps=-0.5)

# 3. Добавим немного "напора" через компрессию (делаем голос плотнее)
y_human = np.clip(y_human * 1.3, -1.0, 1.0)

sf.write("human_anger_v3.wav", y_human, sr)
print("Версия v3 готова. Проверьте: она должна быть быстрее и 'плотнее'.")

Версия v3 готова. Проверьте: она должна быть быстрее и 'плотнее'.


### Silero

In [15]:
import torch
import torchaudio
import pandas as pd
import os

# Используем стандартный торч, он уже стоит в Kaggle
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Запускаемся на {device}")

# Скачиваем стабильную 4-ю версию Silero
local_file = 'model.pt'
if not os.path.exists(local_file):
    torch.hub.download_url_to_file('https://models.silero.ai/models/tts/ru/v4_ru.pt', local_file)

model = torch.package.PackageImporter(local_file).load_pickle("tts_models", "model")
model.to(device)

# Берем ваш файл
df = pd.read_excel("/kaggle/input/datasets/seprenbolat/ozon-test/ozon_test.xlsx")
os.makedirs("silero_audio", exist_ok=True)

# Голос Xenia обычно звучит довольно живо
speaker = 'xenia'
sample_rate = 48000

for i, row in df.iterrows():
    text = str(row['text'])
    emotion = str(row['emotion']).lower().strip()
    out_path = f"silero_audio/sample_{i}_{emotion}.wav"
    
    # Эмулируем эмоции через скорость (rate)
    # Нормальная скорость это где-то 1.0, для гнева говорим быстрее
    speed = 1.0
    if emotion == 'anger':
        speed = 1.2
    elif emotion == 'sadness':
        speed = 0.8
    elif emotion == 'surprise':
        speed = 1.1
        
    try:
        # Генерируем чистый звук
        audio = model.apply_tts(text=text,
                                speaker=speaker,
                                sample_rate=sample_rate,
                                put_accent=True,
                                put_yo=True)
        
        # Если нужна особая скорость, применяем эффекты torchaudio
        if speed != 1.0:
            effects = [['tempo', str(speed)]]
            audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio.unsqueeze(0), sample_rate, effects)
            audio = audio.squeeze(0)
            
        torchaudio.save(out_path, audio.unsqueeze(0), sample_rate)
        print(f"Сделано: {out_path} (эмоция: {emotion})")
        
    except Exception as e:
        print(f"Проблемка на строке {i}: {e}")

print("Всё готово! Аудиофайлы лежат в папке silero_audio")


Запускаемся на cuda


100%|██████████| 38.2M/38.2M [00:05<00:00, 7.49MB/s]


Сделано: silero_audio/sample_0_neutral.wav (эмоция: neutral)
Сделано: silero_audio/sample_1_neutral.wav (эмоция: neutral)
Сделано: silero_audio/sample_2_questioning.wav (эмоция: questioning)
Проблемка на строке 3: module 'torchaudio' has no attribute 'sox_effects'
Проблемка на строке 4: module 'torchaudio' has no attribute 'sox_effects'
Сделано: silero_audio/sample_5_questioning.wav (эмоция: questioning)
Сделано: silero_audio/sample_6_neutral.wav (эмоция: neutral)
Сделано: silero_audio/sample_7_admiration.wav (эмоция: admiration)
Сделано: silero_audio/sample_8_admiration.wav (эмоция: admiration)
Проблемка на строке 9: module 'torchaudio' has no attribute 'sox_effects'
Сделано: silero_audio/sample_10_sarcasm.wav (эмоция: sarcasm)
Проблемка на строке 11: module 'torchaudio' has no attribute 'sox_effects'
Сделано: silero_audio/sample_12_fear.wav (эмоция: fear)
Сделано: silero_audio/sample_13_disgust.wav (эмоция: disgust)
Сделано: silero_audio/sample_14_admiration.wav (эмоция: admiration)


In [16]:
import numpy as np
import librosa
import soundfile as sf # Обычно есть в Kaggle для сохранения wav

# Убедитесь, что librosa установлена (хотя в Kaggle она база)
# !pip install librosa soundfile

for i, row in df.iterrows():
    text = str(row['text'])
    emotion = str(row['emotion']).lower().strip()
    out_path = f"silero_audio/sample_{i}_{emotion}.wav"
    
    # Если файл уже есть и он нейтральный (успешный), можем пропустить, 
    # но лучше пересобрать всё для чистоты
    
    try:
        # 1. Генерируем базу через Silero
        audio_tensor = model.apply_tts(text=text,
                                     speaker=speaker,
                                     sample_rate=sample_rate,
                                     put_accent=True,
                                     put_yo=True)
        
        # Переводим в numpy для librosa
        y = audio_tensor.numpy()
        
        # 2. Настраиваем параметры под эмоции
        rate = 1.0
        volume = 1.0
        
        if emotion == 'anger':
            rate = 1.25    # Быстрее
            volume = 1.4   # Громче и агрессивнее
        elif emotion == 'sadness':
            rate = 0.8     # Медленнее
            volume = 0.7   # Тише, как будто упадок сил
        elif emotion == 'surprise':
            rate = 1.15    # Слегка ускоренно от неожиданности
            volume = 1.2
        elif emotion == 'admiration':
            rate = 1.05    # Чуть бодрее
            volume = 1.1

        # 3. Применяем эффекты через librosa
        if rate != 1.0:
            y = librosa.effects.time_stretch(y, rate=rate)
        
        y = y * volume # Меняем громкость
        
        # 4. Сохраняем результат
        sf.write(out_path, y, sample_rate)
        print(f"Исправлено и сохранено: {out_path} ({emotion})")
        
    except Exception as e:
        # О, я нашел еще одно слабое место... если текст слишком длинный, 
        # Silero может выкинуть ошибку. Но в вашем датасете фразы короткие.
        print(f"Ошибка на строке {i}: {e}")

print("\nТеперь все файлы на месте, включая эмоциональные!")


Исправлено и сохранено: silero_audio/sample_0_neutral.wav (neutral)
Исправлено и сохранено: silero_audio/sample_1_neutral.wav (neutral)
Исправлено и сохранено: silero_audio/sample_2_questioning.wav (questioning)
Исправлено и сохранено: silero_audio/sample_3_anger.wav (anger)
Исправлено и сохранено: silero_audio/sample_4_anger.wav (anger)
Исправлено и сохранено: silero_audio/sample_5_questioning.wav (questioning)
Исправлено и сохранено: silero_audio/sample_6_neutral.wav (neutral)
Исправлено и сохранено: silero_audio/sample_7_admiration.wav (admiration)
Исправлено и сохранено: silero_audio/sample_8_admiration.wav (admiration)
Исправлено и сохранено: silero_audio/sample_9_surprise.wav (surprise)
Исправлено и сохранено: silero_audio/sample_10_sarcasm.wav (sarcasm)
Исправлено и сохранено: silero_audio/sample_11_sadness.wav (sadness)
Исправлено и сохранено: silero_audio/sample_12_fear.wav (fear)
Исправлено и сохранено: silero_audio/sample_13_disgust.wav (disgust)
Исправлено и сохранено: sile

In [17]:
import shutil
import os

# Название вашего будущего архива
archive_name = "ozon_tts_audio_dataset"
# Папка, которую упаковываем
source_dir = "silero_audio"

if os.path.exists(source_dir):
    # Создаем zip-архив
    shutil.make_archive(archive_name, 'zip', source_dir)
    print(f"Готово! Архив создан: {archive_name}.zip")
    print("Теперь вы можете найти его в правой панели Kaggle (раздел Output) и скачать.")
else:
    print(f"Ой, я не нашел папку {source_dir}. Проверьте, завершился ли предыдущий скрипт без ошибок.")


Готово! Архив создан: ozon_tts_audio_dataset.zip
Теперь вы можете найти его в правой панели Kaggle (раздел Output) и скачать.


### F5-tts

### Vits

In [18]:
!apt-get install espeak-ng -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 134 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng-data amd64 1.50+dfsg-10ubuntu0.1 [3,956 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libespeak-ng1 amd64 1.50+dfsg-10ubuntu0.1 [207 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 espeak-ng amd64 1.50+dfsg-

In [19]:
# Ставим только необходимые утилиты для обработки звука
!pip install librosa soundfile pandas --quiet

# Скачиваем сам движок Piper (прекомпилированный)
!wget https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_amd64.tar.gz
!tar -xf piper_amd64.tar.gz

--2026-03-11 22:58:14--  https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_amd64.tar.gz
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/587499842/873099af-72dc-461a-bb2b-670eca1ebd37?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-11T23%3A41%3A43Z&rscd=attachment%3B+filename%3Dpiper_amd64.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-11T22%3A41%3A01Z&ske=2026-03-11T23%3A41%3A43Z&sks=b&skv=2018-11-09&sig=e3dqn0%2FoebjWkvSYiKP66uDWCUEfpIRHaWujztE81H4%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MzI3MTY5NSwibmJmIjoxNzczMjY5ODk1LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi

In [20]:
import os
import zipfile

# 1. Распаковываем наш ZIP с аудио (если вы его уже создали)
zip_path = "ozon_tts_audio_dataset.zip"
data_dir = "dataset/wavs"
os.makedirs(data_dir, exist_ok=True)

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print(f"Распаковано в {data_dir}")

# 2. Переносим метаданные в папку датасета
if os.path.exists("metadata.csv"):
    os.rename("metadata.csv", "dataset/metadata.csv")
    print("Метаданные на месте.")

Распаковано в dataset/wavs
Метаданные на месте.


In [21]:
import pandas as pd
import json

# Загружаем ваш эксель
df = pd.read_excel("/kaggle/input/datasets/seprenbolat/ozon-test/ozon_test.xlsx")

# Создаем файл метаданных для Piper
with open("dataset.jsonl", "w", encoding="utf-8") as f:
    for i, row in df.iterrows():
        emotion = str(row['emotion']).lower().strip()
        line = {
            "audio_path": f"/kaggle/working/silero_audio/sample_{i}_{emotion}.wav",
            "text": str(row['text']),
            "speaker_id": 0 # У нас один голос
        }
        f.write(json.dumps(line, ensure_ascii=False) + "\n")

print("Датасет подготовлен в формате JSONL для Piper!")

Датасет подготовлен в формате JSONL для Piper!


In [22]:
# Устанавливаем зависимости для обучения
!pip install pytorch-lightning==1.9.0 pydantic==1.10.13 librosa --quiet

# Клонируем репозиторий с обучением
!git clone https://github.com/rhasspy/piper.git --quiet
%cd piper/src/python

# Устанавливаем локально
!pip install -e . --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.8/825.8 kB 12.9 MB/s eta 0:00:00a 0:00:01
fatal: destination path 'piper' already exists and is not an empty directory.
[Errno 2] No such file or directory: 'piper/src/python'
/kaggle/working
ERROR: file:///kaggle/working does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [23]:
import json
import os
import pandas as pd

# Загружаем наш эксель
df = pd.read_excel("/kaggle/input/datasets/seprenbolat/ozon-test/ozon_test.xlsx")

# Формируем файл для обучения
output_file = "/kaggle/working/dataset.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for i, row in df.iterrows():
        emotion = str(row['emotion']).lower().strip()
        # Собираем данные в одну строку
        line = {
            "audio_path": f"/kaggle/working/silero_audio/sample_{i}_{emotion}.wav",
            "text": str(row['text']),
            "speaker_id": 0 # У нас один базовый голос
        }
        f.write(json.dumps(line, ensure_ascii=False) + "\n")

print(f"Датасет готов: {output_file}")

Датасет готов: /kaggle/working/dataset.jsonl


In [24]:
# Используем прямую ссылку на LFS (Large File Storage)
!curl -L https://huggingface.co/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU-denis-medium.ckpt?download=true -o base_model.ckpt

# Проверяем, что файл скачался и он не пустой (должен быть > 100Мб)
import os
if os.path.exists("base_model.ckpt") and os.path.getsize("base_model.ckpt") > 1000000:
    print("Ура! Базовая модель скачана успешно.")
else:
    print("Опять 401... Попробуем альтернативную модель (Irina).")
    !curl -L https://huggingface.co/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU-irina-medium.ckpt?download=true -o base_model.ckpt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    29  100    29    0     0    106      0 --:--:-- --:--:-- --:--:--   107
Опять 401... Попробуем альтернативную модель (Irina).
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    29  100    29    0     0    112      0 --:--:-- --:--:-- --:--:--   112


In [28]:
import os
import shutil

# 1. Удаляем старые папки, чтобы не мешались
for folder in ['piper', 'piper-train']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Удалена старая папка {folder}")

# 2. Клонируем ПРАВИЛЬНЫЙ репозиторий через HTTPS
!git clone https://github.com/rhasspy/piper-train.git --quiet
print("Репозиторий piper-train скачан заново.")

# 3. Устанавливаем зависимости (без них Piper не взлетит)
!pip install pytorch-lightning==1.9.0 pydantic==1.10.13 librosa soundfile --quiet
# Устанавливаем сам пакет обучения в систему
!pip install -e /kaggle/working/piper-train --quiet

# 4. Проверяем, на месте ли файл теперь
script_check = "/kaggle/working/piper-train/piper_train/__main__.py"
if os.path.exists(script_check):
    print("Успех! Скрипт найден. Можем начинать.")
else:
    print("Опять пусто... Попробуй запустить !ls -R /kaggle/working/piper-train")

Username for 'https://github.com': ^C
Репозиторий piper-train скачан заново.
ERROR: /kaggle/working/piper-train is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).
Опять пусто... Попробуй запустить !ls -R /kaggle/working/piper-train


In [25]:
import os

# 1. Проверяем, где мы
print(f"Текущая директория: {os.getcwd()}")

# 2. Если папки piper нет, клонируем её заново
if not os.path.exists("/kaggle/working/piper"):
    print("Папка piper не найдена. Клонирую...")
    !git clone https://github.com/rhasspy/piper.git
else:
    print("Папка piper уже на месте.")

# 3. Устанавливаем зависимости (без них не заведется)
!pip install pytorch-lightning==1.9.0 pydantic==1.10.13 librosa soundfile --quiet

# 4. Прописываем жесткие пути к скрипту обучения
# В репозитории Piper путь обычно такой: piper/src/python/piper_train/__main__.py
script_path = "/kaggle/working/piper/src/python/piper_train/__main__.py"

if os.path.exists(script_path):
    print("Скрипт обучения найден! Начинаю запуск...")
    
    # 5. ЗАПУСК ОБУЧЕНИЯ
    # Используем абсолютные пути, чтобы ничего не потерялось
    !python3 {script_path} \
        --dataset /kaggle/working/dataset.jsonl \
        --checkpoint_path /kaggle/working/base_model.ckpt \
        --output_dir /kaggle/working/training_results \
        --batch_size 4 \
        --validation_split 0.1 \
        --num_test_examples 2 \
        --max_epochs 20 \
        --precision 32
else:
    print(f"Критическая ошибка: Файл {script_path} всё еще не найден. Проверьте структуру репозитория через !ls -R piper")

Текущая директория: /kaggle/working
Папка piper уже на месте.
Критическая ошибка: Файл /kaggle/working/piper/src/python/piper_train/__main__.py всё еще не найден. Проверьте структуру репозитория через !ls -R piper


In [26]:
import os

# 1. Вычищаем старый мусор
!rm -rf /kaggle/working/piper

# 2. Клонируем ПРАВИЛЬНЫЙ репозиторий для обучения
!git clone https://github.com/rhasspy/piper-train.git --quiet
print("Репозиторий piper-train скачан.")

# 3. Устанавливаем зависимости именно для обучения
!pip install pytorch-lightning==1.9.0 pydantic==1.10.13 librosa soundfile --quiet
# Устанавливаем сам пакет в режиме редактирования
!pip install -e /kaggle/working/piper-train --quiet

# 4. Устанавливаем системный espeak-ng (критично для фонем!)
!apt-get update && apt-get install espeak-ng -y --quiet

Username for 'https://github.com': ^C
Репозиторий piper-train скачан.
^C
ERROR: Operation cancelled by user
ERROR: /kaggle/working/piper-train is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 http://sec

In [27]:
import sys
# Добавляем путь в систему, на всякий случай
sys.path.append("/kaggle/working/piper-train")

# Назначаем путь к скрипту
train_script = "/kaggle/working/piper-train/piper_train/__main__.py"

if os.path.exists(train_script):
    print("Скрипт найден! Зажигаем печь обучения...")
    
    !python3 {train_script} \
        --dataset /kaggle/working/dataset.jsonl \
        --checkpoint_path /kaggle/working/base_model.ckpt \
        --output_dir /kaggle/working/training_results \
        --batch_size 4 \
        --validation_split 0.05 \
        --num_test_examples 1 \
        --max_epochs 15 \
        --precision 32
else:
    print("Что-то пошло не так с установкой piper-train. Проверь вывод ячейки выше.")

Что-то пошло не так с установкой piper-train. Проверь вывод ячейки выше.
